# Mangrove FN vs HLS Raster (Paper 3)

This notebook compares values inside vs outside Forces of Nature (FN) mangrove polygons.

The raster is selected automatically from `dphil_paper_3/inputs/ndvi` using this priority:
- `HLS_masked_median_composite_2months_before_epsg3448*.tif` (if present)
- otherwise `HLS_masked_NDVI_2months_before_epsg3448*.tif`

It includes:
- Raster and vector metadata checks
- Quicklook with mangrove overlay
- Per-band (or NDVI) summary statistics for mangrove and non-mangrove pixels
- Sampled distribution plots for visual comparison


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import geometry_mask
from rasterio.enums import Resampling
from rasterio.transform import array_bounds, Affine
import matplotlib.pyplot as plt
from shapely.geometry import box

plt.style.use('default')
pd.set_option('display.max_columns', 100)


In [ ]:
def find_project_root(start: Path) -> Path:
    """Walk up from current working directory until a folder containing dphil_papers is found."""
    start = start.resolve()
    candidates = [start] + list(start.parents)
    for p in candidates:
        if (p / 'dphil_papers').exists():
            return p
    raise FileNotFoundError(
        f'Could not find project root containing dphil_papers from: {start}'
    )

ROOT = find_project_root(Path.cwd())
ndvi_dir = ROOT / 'dphil_papers/dphil_paper_3/inputs/ndvi'

def pick_primary_raster() -> Path:
    patterns = [
        'HLS_masked_median_composite_2months_before_epsg3448*.tif',
        'HLS_masked_NDVI_2months_before_epsg3448*.tif',
    ]
    for pattern in patterns:
        matches = sorted(ndvi_dir.glob(pattern))
        if len(matches) == 1:
            return matches[0]
        if len(matches) > 1:
            names = '\n'.join(str(m.name) for m in matches)
            raise FileExistsError(f'Ambiguous raster pattern {pattern} matched multiple files:\n{names}')
    raise FileNotFoundError(
        f'No supported raster found in {ndvi_dir}. Tried patterns: {patterns}'
    )

raster_path = pick_primary_raster()
raster_mode = 'composite' if 'median_composite' in raster_path.name else 'ndvi'
map_title_prefix = 'HLS Composite' if raster_mode == 'composite' else 'HLS NDVI'
mangrove_fn_path = ROOT / 'dphil_papers/dphil_paper_3/inputs/forces_of_nature_mangroves/mangroves.shp'

# Optional alternative source:
# mangrove_fn_path = ROOT / 'dphil_papers/dphil_common_cross_cutting/common_incoming_data/landcover/mangroves_fn/MangrovesFN.gpkg'

print(f'Working directory: {Path.cwd()}')
print(f'Project root: {ROOT}')
print(f'Raster mode: {raster_mode}')
print(f'Raster exists: {raster_path.exists()}')
print(f'Mangrove file exists: {mangrove_fn_path.exists()}')
print(raster_path)
print(mangrove_fn_path)


In [ ]:
# Diagnostic open test
try:
    with rasterio.open(raster_path) as src:
        print('Raster opened successfully')
        print('CRS:', src.crs)
        print('Size:', src.width, 'x', src.height)
        print('Bands:', src.count, src.descriptions)
except Exception as e:
    print('Raster open failed:')
    print(type(e).__name__, e)
    raise


In [ ]:
with rasterio.open(raster_path) as src:
    raster_summary = {
        'crs': str(src.crs),
        'width': src.width,
        'height': src.height,
        'count': src.count,
        'dtype': src.dtypes[0],
        'nodata': src.nodata,
        'bounds': src.bounds,
        'descriptions': src.descriptions,
        'resolution': src.res,
    }

pd.Series(raster_summary)


In [ ]:
mangroves = gpd.read_file(mangrove_fn_path)

with rasterio.open(raster_path) as src:
    raster_crs = src.crs
    raster_bounds = src.bounds

if mangroves.crs != raster_crs:
    mangroves = mangroves.to_crs(raster_crs)

mangroves = mangroves[mangroves.geometry.notnull()].copy()
mangroves = mangroves[~mangroves.geometry.is_empty].copy()
mangroves = mangroves[mangroves.geometry.is_valid].copy()

raster_extent = gpd.GeoDataFrame(geometry=[box(*raster_bounds)], crs=raster_crs)
mangroves_in_extent = gpd.overlay(mangroves, raster_extent, how='intersection')

print(f'Total mangrove features: {len(mangroves):,}')
print(f'Mangrove features intersecting raster extent: {len(mangroves_in_extent):,}')
print(f'Total mangrove area in extent (ha): {mangroves_in_extent.geometry.area.sum() / 10_000:,.2f}')

mangroves_in_extent.head(3)


In [ ]:
def robust_stretch(multiband_array, low=2, high=98):
    stretched = np.zeros_like(multiband_array, dtype='float32')
    for i in range(multiband_array.shape[0]):
        band = multiband_array[i]
        valid = np.isfinite(band)
        if not valid.any():
            continue
        lo, hi = np.percentile(band[valid], [low, high])
        denom = (hi - lo) if (hi - lo) != 0 else 1.0
        stretched[i] = np.clip((band - lo) / denom, 0, 1)
    return stretched

with rasterio.open(raster_path) as src:
    downsample = 8
    out_h = max(1, src.height // downsample)
    out_w = max(1, src.width // downsample)

    if src.count >= 3:
        # Composite convention: B2 (blue), B3 (green), B4 (red) -> RGB as [3,2,1]
        rgb = src.read([3, 2, 1], out_shape=(3, out_h, out_w), resampling=Resampling.bilinear)
        quicklook_title = f'{map_title_prefix} RGB (B4/B3/B2) with FN mangrove boundaries'
    else:
        # NDVI fallback: render grayscale as pseudo-RGB for consistent plotting pipeline
        single = src.read(1, out_shape=(out_h, out_w), resampling=Resampling.bilinear)
        rgb = np.stack([single, single, single], axis=0)
        quicklook_title = f'{map_title_prefix} (single-band grayscale) with FN mangrove boundaries'

    transform_small = src.transform * Affine.scale(src.width / out_w, src.height / out_h)

rgb_stretched = robust_stretch(rgb)
left, bottom, right, top = array_bounds(out_h, out_w, transform_small)

fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(np.moveaxis(rgb_stretched, 0, -1), extent=[left, right, bottom, top], origin='upper')
mangroves_in_extent.boundary.plot(ax=ax, linewidth=0.5, color='cyan', alpha=0.8)
ax.set_title(quicklook_title)
ax.set_xlabel('Easting (m)')
ax.set_ylabel('Northing (m)')
plt.show()


In [ ]:
def sample_pixels(values, n=150_000, seed=42):
    if values.size <= n:
        return values
    rng = np.random.default_rng(seed)
    idx = rng.choice(values.size, size=n, replace=False)
    return values[idx]

summary_rows = []
sample_rows = []

with rasterio.open(raster_path) as src:
    if src.count >= 3:
        band_labels = {1: 'B2_blue', 2: 'B3_green', 3: 'B4_red'}
    else:
        band_name = src.descriptions[0] if (src.descriptions and src.descriptions[0]) else 'NDVI'
        band_labels = {1: str(band_name)}

    mangrove_mask = geometry_mask(
        [geom for geom in mangroves_in_extent.geometry if geom is not None and not geom.is_empty],
        transform=src.transform,
        out_shape=(src.height, src.width),
        invert=True,
    )

    for band_idx, band_name in band_labels.items():
        arr = src.read(band_idx)
        valid = np.isfinite(arr)

        vals_m = arr[valid & mangrove_mask]
        vals_n = arr[valid & (~mangrove_mask)]

        for group_name, values in [('mangrove', vals_m), ('non_mangrove', vals_n)]:
            if values.size == 0:
                continue

            summary_rows.append({
                'band': band_name,
                'group': group_name,
                'count': int(values.size),
                'mean': float(np.mean(values)),
                'median': float(np.median(values)),
                'std': float(np.std(values)),
                'p05': float(np.percentile(values, 5)),
                'p25': float(np.percentile(values, 25)),
                'p75': float(np.percentile(values, 75)),
                'p95': float(np.percentile(values, 95)),
            })

            sampled = sample_pixels(values, n=150_000, seed=band_idx * (7 if group_name == 'mangrove' else 11))
            sample_rows.append(pd.DataFrame({
                'band': band_name,
                'group': group_name,
                'value': sampled,
            }))

summary_df = pd.DataFrame(summary_rows)
samples_df = pd.concat(sample_rows, ignore_index=True)

print(f'Mangrove pixel count: {summary_df[summary_df.group == "mangrove"]["count"].sum():,}')
print(f'Non-mangrove pixel count: {summary_df[summary_df.group == "non_mangrove"]["count"].sum():,}')
summary_df


In [ ]:
summary_pivot = (
    summary_df
    .pivot(index='band', columns='group', values=['mean', 'median', 'std', 'p05', 'p95'])
    .round(4)
)
summary_pivot


In [ ]:
bands = summary_df['band'].drop_duplicates().tolist()
fig, axes = plt.subplots(1, len(bands), figsize=(max(6, 5 * len(bands)), 4.5), sharey=False)

if len(bands) == 1:
    axes = [axes]

for ax, band in zip(axes, bands):
    s_m = samples_df[(samples_df['band'] == band) & (samples_df['group'] == 'mangrove')]['value'].to_numpy()
    s_n = samples_df[(samples_df['band'] == band) & (samples_df['group'] == 'non_mangrove')]['value'].to_numpy()

    if s_m.size == 0 or s_n.size == 0:
        ax.set_visible(False)
        continue

    q_low = np.percentile(np.concatenate([s_m, s_n]), 1)
    q_high = np.percentile(np.concatenate([s_m, s_n]), 99)
    bins = np.linspace(q_low, q_high, 80)

    ax.hist(s_n, bins=bins, alpha=0.45, label='non_mangrove', density=True)
    ax.hist(s_m, bins=bins, alpha=0.55, label='mangrove', density=True)
    ax.set_title(band)
    ax.set_xlabel('Pixel value')

axes[0].set_ylabel('Density')
axes[-1].legend(loc='upper right')
fig.suptitle('Value distributions: mangrove vs non-mangrove')
plt.tight_layout()
plt.show()


## Notes
- This TIFF contains `B2`, `B3`, `B4`, and `Fmask` bands, so this notebook compares visible bands only.
- If you want NDVI, you will need a NIR band (for example `B8`/`B5`) and can then compute `(NIR - Red) / (NIR + Red)`.
- To compare against a different FN mangrove file, just change `mangrove_fn_path` and rerun.


In [ ]:
# Export a presentation-ready map: HLS + FN mangroves with clear legend
import matplotlib.patches as mpatches

output_png = ROOT / 'dphil_papers/dphil_paper_3/results/threats/ndvi/draft_processed_images/fn_mangroves_on_hls_with_legend_from_notebook.png'

fig, ax = plt.subplots(figsize=(10, 8), constrained_layout=True)
ax.imshow(np.moveaxis(rgb_stretched, 0, -1), extent=[left, right, bottom, top], origin='upper')

overlay_fill = '#ffd400'   # bright yellow
overlay_edge = '#111111'   # black
mangroves_in_extent.plot(ax=ax, color=overlay_fill, edgecolor=overlay_edge, linewidth=0.35, alpha=0.55)

ax.set_title(f'{map_title_prefix} with FN Mangroves Overlay')
ax.set_xlabel('Easting (m, EPSG:3448)')
ax.set_ylabel('Northing (m, EPSG:3448)')
ax.set_aspect('equal')

legend_patch = mpatches.Patch(facecolor=overlay_fill, edgecolor=overlay_edge, label='FN mangrove polygons')
ax.legend(handles=[legend_patch], loc='lower left', frameon=True, framealpha=0.95)

fig.savefig(output_png, dpi=300)
plt.show()
print('Saved:', output_png)


## Forest Overlay From Landcover (Robyn classes)

This section overlays forest-related classes from the landcover map on top of the selected raster (composite if available, otherwise NDVI), using definitions from `Robyn_catchment_analysis.py`:
- `forest_flood_equivalent_classes`
- `mixed_land_use_fractions` (mixed classes get fractional forest-equivalent accounting)


In [ ]:
import importlib.util
import matplotlib.patches as mpatches

landcover_path = ROOT / 'dphil_papers/dphil_common_cross_cutting/common_incoming_data/landcover/2013_landcover/2013_landuse_landcover.gpkg'
robyn_defs_path = ROOT / 'dphil_papers/robyns_libraries/Robyn_catchment_analysis.py'

spec = importlib.util.spec_from_file_location('robyn_ca', robyn_defs_path)
robyn_ca = importlib.util.module_from_spec(spec)
spec.loader.exec_module(robyn_ca)

forest_classes = set(robyn_ca.forest_flood_equivalent_classes)
mixed_class_fractions = dict(robyn_ca.mixed_land_use_fractions)
mixed_classes = set(mixed_class_fractions.keys())
forest_only_classes = forest_classes - mixed_classes

landcover = gpd.read_file(landcover_path, columns=['Classify', 'geometry'])
if landcover.crs != raster_crs:
    landcover = landcover.to_crs(raster_crs)

landcover = landcover[landcover.geometry.notnull() & ~landcover.geometry.is_empty].copy()
forest_lc = landcover[landcover['Classify'].isin(forest_classes)].copy()
forest_only_lc = forest_lc[forest_lc['Classify'].isin(forest_only_classes)].copy()
mixed_lc = forest_lc[forest_lc['Classify'].isin(mixed_classes)].copy()

# Keep plotting focused around the HLS extent
extent_poly = box(*raster_bounds)
forest_only_lc = forest_only_lc[forest_only_lc.intersects(extent_poly)].copy()
mixed_lc = mixed_lc[mixed_lc.intersects(extent_poly)].copy()


def forest_fraction_from_mixed_class(class_name: str) -> float:
    parts = mixed_class_fractions.get(class_name, {})
    # Works for both naming schemes in Robyn_catchment_analysis.py
    return float(sum(v for k, v in parts.items() if 'forest' in str(k).lower()))

mixed_lc['forest_fraction'] = mixed_lc['Classify'].map(forest_fraction_from_mixed_class).fillna(0.0)

forest_only_ha = forest_only_lc.geometry.area.sum() / 10_000
mixed_forest_equiv_ha = (mixed_lc.geometry.area * mixed_lc['forest_fraction']).sum() / 10_000
total_forest_equiv_ha = forest_only_ha + mixed_forest_equiv_ha

print('Landcover path:', landcover_path)
print(f'Forest-only classes polygons: {len(forest_only_lc):,}')
print(f'Mixed classes polygons: {len(mixed_lc):,}')
print(f'Forest-only area (ha): {forest_only_ha:,.2f}')
print(f'Mixed forest-equivalent area (ha): {mixed_forest_equiv_ha:,.2f}')
print(f'Total forest-equivalent area (ha): {total_forest_equiv_ha:,.2f}')


In [ ]:
# Visual overlay: HLS RGB + forest-related landcover classes

forest_overlay_output = ROOT / 'dphil_papers/dphil_paper_3/results/threats/ndvi/draft_processed_images/hls_landcover_forest_overlay_from_robyn_classes.png'

fig, ax = plt.subplots(figsize=(10, 8), constrained_layout=True)
ax.imshow(np.moveaxis(rgb_stretched, 0, -1), extent=[left, right, bottom, top], origin='upper')

forest_color = '#00b050'  # clear green
mixed_color = '#ff9f1c'   # orange for mixed classes

if len(forest_only_lc) > 0:
    forest_only_lc.plot(ax=ax, color=forest_color, edgecolor='none', alpha=0.50)
if len(mixed_lc) > 0:
    mixed_lc.plot(ax=ax, color=mixed_color, edgecolor='none', alpha=0.45)

legend_items = [
    mpatches.Patch(facecolor=forest_color, edgecolor='none', alpha=0.50, label='Forest classes (full)'),
    mpatches.Patch(facecolor=mixed_color, edgecolor='none', alpha=0.45, label='Mixed classes (fractional forest-equivalent)'),
]
ax.legend(handles=legend_items, loc='lower left', frameon=True, framealpha=0.95)

ax.set_title(f'{map_title_prefix} with Forest-related Landcover Overlay')
ax.set_xlabel('Easting (m, EPSG:3448)')
ax.set_ylabel('Northing (m, EPSG:3448)')
ax.set_aspect('equal')

fig.savefig(forest_overlay_output, dpi=300)
plt.show()
print('Saved:', forest_overlay_output)


In [ ]:
# Export separate maps: forest-only and mixed-only overlays

forest_only_output = ROOT / 'dphil_papers/dphil_paper_3/results/threats/ndvi/draft_processed_images/hls_forest_only_overlay_from_robyn_classes.png'
mixed_only_output = ROOT / 'dphil_papers/dphil_paper_3/results/threats/ndvi/draft_processed_images/hls_mixed_only_overlay_from_robyn_classes.png'

# Forest-only
fig, ax = plt.subplots(figsize=(10, 8), constrained_layout=True)
ax.imshow(np.moveaxis(rgb_stretched, 0, -1), extent=[left, right, bottom, top], origin='upper')
forest_color = '#00b050'
if len(forest_only_lc) > 0:
    forest_only_lc.plot(ax=ax, color=forest_color, edgecolor='none', alpha=0.55)
ax.legend(handles=[
    mpatches.Patch(facecolor=forest_color, edgecolor='none', alpha=0.55, label='Forest classes (full)')
], loc='lower left', frameon=True, framealpha=0.95)
ax.set_title(f'{map_title_prefix} with Forest Classes Only')
ax.set_xlabel('Easting (m, EPSG:3448)')
ax.set_ylabel('Northing (m, EPSG:3448)')
ax.set_aspect('equal')
fig.savefig(forest_only_output, dpi=300)
plt.show()

# Mixed-only
fig, ax = plt.subplots(figsize=(10, 8), constrained_layout=True)
ax.imshow(np.moveaxis(rgb_stretched, 0, -1), extent=[left, right, bottom, top], origin='upper')
mixed_color = '#ff9f1c'
if len(mixed_lc) > 0:
    mixed_lc.plot(ax=ax, color=mixed_color, edgecolor='none', alpha=0.50)
ax.legend(handles=[
    mpatches.Patch(facecolor=mixed_color, edgecolor='none', alpha=0.50, label='Mixed land-use classes')
], loc='lower left', frameon=True, framealpha=0.95)
ax.set_title(f'{map_title_prefix} with Mixed Land-use Classes Only')
ax.set_xlabel('Easting (m, EPSG:3448)')
ax.set_ylabel('Northing (m, EPSG:3448)')
ax.set_aspect('equal')
fig.savefig(mixed_only_output, dpi=300)
plt.show()

print('Saved:', forest_only_output)
print('Saved:', mixed_only_output)
